Importación de modulos necesarios

In [ ]:
import cv2  
import math 
import csv

from ultralytics import YOLO

**TAREA 1**

Carga y configuración de los modelos.

In [2]:
source_video = "C0142.mp4"
output_video = "output.mp4"
csv_output = "detections.csv"

# Carga de los modelos: preentrenado y detección de matrículas
general_model = YOLO("yolo11n.pt")
plate_model = YOLO("./runs/detect/yolo_runs/plates_detection/weights/best.pt")

# Definición de las clases de interés
PERSON = 0
VEHICLES = {1, 2, 3, 5} # bicicleta, coche, moto, guagua
CLASSES = [PERSON, *VEHICLES]
PLATE_VEHICLES = {2, 3, 5} # vehículos con matrícula

# Parámetros de detección y recorte
conf_coco = 0.4
conf_plate = 0.3
padding = 0.05 # margen extra alrededor del vehículo (5%)

# Color por clase (BGR) y color de matrícula
class_colors = {
    0: (0, 200, 0), # persona
    1: (200, 0, 200), # bicicleta
    2: (255, 120, 0), # coche
    3: (0, 165, 255), # moto
    5: (0, 200, 255), # guagua
}
plate_color = (0, 0, 255)

# Nombres para mostrar en pantalla
display_names = {0: "Persona", 1: "Bicicleta", 2: "Coche", 3: "Moto", 5: "Guagua"}

Se definen funciones auxiliares para la generación del archivo CSV

In [3]:
def write_csv_header(path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
              "x1", "y1", "x2", "y2", "matricula_en_su_caso", "confianza_matricula",
              "mx1", "my1", "mx2", "my2", "texto_matricula"])


def make_row(frame_n, name, conf, tid, box, plate=None):
    """plate: None o (pconf, mx1, my1, mx2, my2)."""
    x1, y1, x2, y2 = box
    if plate is not None:
        pconf, mx1, my1, mx2, my2 = plate
        plate_part = ["Si", f"{pconf:.3f}", mx1, my1, mx2, my2, ""]
    else:
        plate_part = ["No", "", "", "", "", "", ""]
    return [frame_n, name, f"{conf:.3f}", tid if tid is not None else "",
            x1, y1, x2, y2, *plate_part]


def append_rows(path, rows):
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)

Configuración para preparar el vídeo

In [4]:
vid = cv2.VideoCapture(source_video)
fps = vid.get(cv2.CAP_PROP_FPS)
w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

write_csv_header(csv_output)

unique_ids = {c: set() for c in CLASSES}
track_class = {}
plate_ids = set()

Se define la función auxiliar draw, para dibujar las cajas de las detecciones.

In [5]:
def draw(frame, box, label, color):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

In [ ]:
frame_n = 0
while True:
    ret, frame = vid.read()
    if not ret:
        break
    frame_n += 1

    r = general_model.track(frame, persist=True, classes=CLASSES,
                            conf=conf_coco, verbose=True)[0]
    ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
           else [None] * len(r.boxes))

    rows = []
    for box, cls, conf, tid in zip(r.boxes.xyxy.cpu().numpy(),
                                   r.boxes.cls.cpu().numpy().astype(int),
                                   r.boxes.conf.cpu().numpy(), ids):
        x1, y1, x2, y2 = map(int, box)

        if tid is not None:
            cls = track_class.setdefault(tid, cls)
            unique_ids[cls].add(tid)

        name = general_model.names[cls]
        tag = f"{name} #{tid}" if tid is not None else name
        draw(frame, box, f"{tag} {conf:.2f}", class_colors[cls])

        plate = None
        if cls in PLATE_VEHICLES:
            bw, bh = x2 - x1, y2 - y1
            cx1 = max(0, int(x1 - bw * padding)); cy1 = max(0, int(y1 - bh * padding))
            cx2 = min(w, int(x2 + bw * padding)); cy2 = min(h, int(y2 + bh * padding))
            crop = frame[cy1:cy2, cx1:cx2]
            if crop.size:
                res = plate_model.predict(crop, conf=conf_plate, verbose=True)[0]
                if len(res.boxes):
                    i = int(res.boxes.conf.argmax())
                    pconf = float(res.boxes.conf[i])
                    px1, py1, px2, py2 = res.boxes.xyxy[i].cpu().numpy()
                    plate = (pconf, int(px1 + cx1), int(py1 + cy1),
                             int(px2 + cx1), int(py2 + cy1))
                    draw(frame, plate[1:], f"plate {pconf:.2f}", plate_color)
                    if tid is not None:
                        plate_ids.add(tid)

        rows.append(make_row(frame_n, name, conf, tid, (x1, y1, x2, y2), plate))

    append_rows(csv_output, rows)

    lines = [f"{display_names[c]}: {len(unique_ids[c])}" for c in CLASSES]
    lines.append(f"matricula: {len(plate_ids)}")
    for i, txt in enumerate(lines):
        cv2.putText(frame, txt, (10, 25 + i * 22), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2, cv2.LINE_AA)

    out.write(frame)

vid.release()
out.release()
print("Vídeo con las detecciones generado:", output_video)
print("CSV con las detecciones generado:", csv_output)

**TAREA 2**